<a href="https://colab.research.google.com/github/kamalrawat77/agentic-iam-lab/blob/main/week07-agentic-rag/Nugget039_Self_Healing_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Nugget 039: Self-Healing Agent (Error Handling & Recovery)

In [10]:
!pip install -q google-genai
import json

In [106]:
from google import genai

def callGPT(prompt):
  client = genai.Client(api_key="APIKEY")
  response = client.models.generate_content(
      model="gemini-2.5-flash",
      contents=prompt
  )

  return response

In [17]:
def cleanse_response(response):
  clean_response=clean_response = response.text
  clean_response = clean_response.replace("```json", "")
  clean_response = clean_response.replace("```", "")
  clean_response = clean_response.strip()

  return clean_response

In [18]:
def return_json(clean_response,objname):
  responseObj = json.loads(clean_response)
  if not objname:
    return responseObj
  resJsonObj  = responseObj[objname]
  return resJsonObj



Tool List

In [111]:
def dormant_accounts():

    return "Dormant Accounts: 27"


def department_breakdown():

    return """
IT: 45
HR: 12
Finance: 18
"""


def trend_analysis():
    return "Dormant accounts increased from 20 to 27"

def search_history():
    #return "Past investigation found delayed terminations"
    raise Exception(
        "Database unavailable"
    )

Tool Registry

In [24]:
tools = {
    "dormant_accounts": dormant_accounts,
    "department_breakdown": department_breakdown,
    "search_history": search_history,
    "trend_analysis": trend_analysis
}

Create memory list

In [115]:
memory = []

In [116]:
try:

    result = search_history()

except Exception as e:

    result = f"ERROR: {str(e)}"

In [117]:
print(result)

ERROR: Database unavailable


In [118]:
memory.append({
    "tool":"search_history",
    "status":"failed",
    "observation":result
})

In [74]:
question = """
Why are dormant accounts increasing?
"""

Planner prompt

In [119]:
planner_prompt = f"""
Question:

{question}

Investigation Memory:

{memory}

Available Tools:

- search_history
- trend_analysis

If a tool failed,
choose another tool.

Return JSON:

{{
  "action":"tool_name"
}}
"""

In [120]:
response=callGPT(planner_prompt)

In [121]:
toolDecision=return_json(cleanse_response(response),None)

In [122]:
print(toolDecision)

{'action': 'trend_analysis'}


In [123]:
tool_name = toolDecision["action"]

result = tools[tool_name]()

In [124]:
memory.append({
    "tool": tool_name,
    "observation": result
})

In [125]:
print(memory)

[{'tool': 'search_history', 'status': 'failed', 'observation': 'ERROR: Database unavailable'}, {'tool': 'trend_analysis', 'observation': 'Dormant accounts increased from 20 to 27'}]
